# VibeScent Week 4 — Demo Day Notebook

**Owner:** Harsh Agarwal  
**Pipeline version:** `w4v1`  
**Purpose:** Serve the full 4-signal fusion + optional reranker pipeline over FastAPI, expose publicly via Cloudflare Tunnel (+ ngrok backup), and hand Darren a diff he can drop into his Next.js route.

---

## Pre-Demo Runbook Checklist

Run through this list **before** opening the notebook in front of an audience.

- [ ] Colab runtime: **A100 GPU** selected (`Runtime > Change runtime type`)
- [ ] Google Drive mounted and `vibescent/` directory visible at `DRIVE_BASE`
- [ ] `week3/best_weights.json` and `week3/reranker_decision.json` exist on Drive
- [ ] `week2/` embeddings `.npy` files and enriched CSV exist on Drive
- [ ] `GEMINI_API_KEY` stored in Colab Secrets (`🔑` sidebar) — needed only if reranker shipped
- [ ] `VOYAGE_API_KEY` stored in Colab Secrets (used for text-query embedding fallback)
- [ ] `cloudflared` binary installs cleanly (cell 2 pip install)
- [ ] Run **all cells top-to-bottom** once as a dry-run 30 min before demo
- [ ] Paste both tunnel URLs into Darren's `.env.local` as `NEXT_PUBLIC_BACKEND_URL`
- [ ] Verify `artifacts/week4/locked_responses.json` saved (Stage 7)
- [ ] Smoke-test cell returns HTTP 200 with correct JSON shape (Stage 9)

---

## Stage Map

| Stage | Cell(s) | Description |
|-------|---------|-------------|
| 1 | Setup | Clone repo, install deps, mount Drive, detect GPU |
| 2 | Load W3 Artifacts | Fusion weights, reranker decision, embeddings, DataFrame |
| 3 | Load Models | Qwen3-Embedding-8B, Qwen3-VL-Embedding-8B, Neil CNN, optional VL-32B reranker |
| 4 | Build Engine | `VibeScentEngine` class conforming to `RecommendationEngine` protocol |
| 5 | Start FastAPI | `create_app(engine)` + uvicorn in background thread |
| 6 | Dual Tunnel | Cloudflare primary + ngrok backup |
| 7 | Demo Cache | Pre-generate 5 locked responses, persist to Drive |
| 8 | Frontend Patch | Emit `frontend_patch.diff` for Darren |
| 9 | Smoke Test | POST to both tunnels, assert contract shape |
| 10 | Pre-Demo Runbook | Printed day-of instructions |

In [ ]:
# Stage 1: Setup -- uv, depth-1 clone, parallel Drive+secrets, GPU detection, HF cache
import os, sys, time, subprocess, threading
from pathlib import Path
_t0 = time.perf_counter()
PIPELINE_VERSION = 'w4v1'
REPO_URL='https://github.com/Harsh-Agarwal0/vibescent.git'; REPO_DIR='/content/vibescent'
FASTAPI_PORT=8000; FASTAPI_HOST='127.0.0.1'; _BASE_URL=f'http://{FASTAPI_HOST}:{FASTAPI_PORT}'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','uv'],check=True,capture_output=True)
print('uv ready')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--depth=1','--single-branch','--branch=Text_Processing',REPO_URL,REPO_DIR],check=True,capture_output=True)
    print('Repo cloned')
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],capture_output=True); print('Repo up to date')
os.chdir(REPO_DIR)
from tqdm.auto import tqdm
bar = tqdm(['Drive','Secrets','Pkg','GPU deps','GPU'],desc='Stage 1 -- Setup',ncols=100,unit='step')
_drive_exc=None
def _mount():
    global _drive_exc,DRIVE_BASE,W2_ARTIFACTS,W3_ARTIFACTS,W4_ARTIFACTS,HF_CACHE,CLOUDFLARED_BIN
    try:
        from google.colab import drive; drive.mount('/content/drive',force_remount=False)
        DRIVE_BASE='/content/drive/MyDrive/vibescent'; HF_CACHE=str(Path(DRIVE_BASE).parent/'hf_cache')
    except Exception as e:
        _drive_exc=e; DRIVE_BASE=f'{REPO_DIR}/artifacts'; HF_CACHE='/tmp/hf_cache'
    W2_ARTIFACTS=DRIVE_BASE; W3_ARTIFACTS=f'{DRIVE_BASE}/week3'; W4_ARTIFACTS=f'{DRIVE_BASE}/week4'
    CLOUDFLARED_BIN='/usr/local/bin/cloudflared'
    os.makedirs(W3_ARTIFACTS,exist_ok=True); os.makedirs(W4_ARTIFACTS,exist_ok=True)
    os.environ['HF_HOME']=HF_CACHE; os.makedirs(HF_CACHE,exist_ok=True)
_dt=threading.Thread(target=_mount,daemon=False); _dt.start()
bar.update(1); bar.set_description('Secrets')
_gemini_key=_voyage_key=None
def _secrets():
    global _gemini_key,_voyage_key
    def _g(n):
        try:
            from google.colab import userdata; return userdata.get(n)
        except: return os.environ.get(n)
    _gemini_key,_voyage_key=_g('GEMINI_API_KEY'),_g('VOYAGE_API_KEY')
_st=threading.Thread(target=_secrets,daemon=False); _st.start()
bar.update(1); bar.set_description('Pkg')
subprocess.run(['uv','pip','install','--system','-q','-e',REPO_DIR],check=True,capture_output=True)
bar.update(1); bar.set_description('GPU deps')
_DEPS=['torch>=2.4,<3.0','sentence-transformers>=3.4,<4.0','transformers>=4.57,<5.0',
       'accelerate>=1.3,<2.0','fastapi>=0.115,<1.0','uvicorn[standard]>=0.34,<1.0',
       'httpx>=0.27,<1.0','pyngrok>=7.2,<8.0','nest-asyncio>=1.6,<2.0','Pillow>=10.0,<12.0','tqdm']
subprocess.run(['uv','pip','install','--system','-q']+_DEPS,check=True,capture_output=True)
_dt.join(); _st.join()
bar.update(1); bar.set_description('GPU')
if not os.path.isfile(CLOUDFLARED_BIN):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O',CLOUDFLARED_BIN],capture_output=True)
    subprocess.run(['chmod','+x',CLOUDFLARED_BIN],capture_output=True)
import torch
GPU_TIER='CPU'; DEVICE='cpu'
if torch.cuda.is_available():
    DEVICE='cuda'; _v=torch.cuda.get_device_properties(0).total_memory/1e9
    GPU_TIER='A100' if _v>=35 else('L4' if _v>=20 else 'T4')
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM={_v:.1f}GB  tier={GPU_TIER}')
else: print('WARNING: No CUDA GPU')
bar.update(1); bar.close()
if _gemini_key: os.environ['GEMINI_API_KEY']=_gemini_key; print('GEMINI_API_KEY loaded')
else: print('WARNING: GEMINI_API_KEY not set')
if _voyage_key: os.environ['VOYAGE_API_KEY']=_voyage_key
if _drive_exc:
    print('='*60); print(f'WARNING: Drive mount failed! ({_drive_exc})')
    print(f'Artifacts -> {DRIVE_BASE}  (LOST on disconnect!)'); print('='*60)
print(f'Setup {time.perf_counter()-_t0:.1f}s | DRIVE={DRIVE_BASE} | GPU={GPU_TIER}')

In [ ]:
# Stage 2: Load Week 2 + 3 Artifacts -- parallel .npy loads
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import json,os,time; import numpy as np; import pandas as pd
from concurrent.futures import ThreadPoolExecutor,as_completed
from tqdm.auto import tqdm
_t0=time.perf_counter()
def _req(p,l):
    if not os.path.exists(p): raise FileNotFoundError(f'[REQUIRED] {l} not found:\n  {p}\nRun Week 3 first.')
    return p
_req(os.path.join(W3_ARTIFACTS,'best_weights.json'),'W3 best_weights.json')
_req(os.path.join(W3_ARTIFACTS,'reranker_decision.json'),'W3 reranker_decision.json')
with open(os.path.join(W3_ARTIFACTS,'best_weights.json')) as fh: _w=json.load(fh)
FUSION_WEIGHTS=_w.get('weights',_w)
for _k in ('text','multimodal','image','structured'): FUSION_WEIGHTS.setdefault(_k,0.25)
_tot=sum(FUSION_WEIGHTS.values()); FUSION_WEIGHTS={k:v/_tot for k,v in FUSION_WEIGHTS.items()}
print(f'Fusion weights: {FUSION_WEIGHTS}')
with open(os.path.join(W3_ARTIFACTS,'reranker_decision.json')) as fh: _d=json.load(fh)
SHIP_RERANKER=bool(_d.get('ship_reranker',False))
print(f'Reranker: ship={SHIP_RERANKER}  reason={_d.get("reason","n/a")!r}')
# W2 artifact paths (Week 2 saves directly to DRIVE_BASE, not a week2/ subfolder)
TEXT_EMB_PATH=os.path.join(W2_ARTIFACTS,'fragrance_raw_full','embeddings.npy')
ENR_EMB_PATH=os.path.join(W2_ARTIFACTS,'fragrance_enriched_2k','embeddings.npy')
MM_EMB_PATH=os.path.join(W2_ARTIFACTS,'multimodal_2k','doc_embeddings.npy')
_req(TEXT_EMB_PATH,'W2 raw text embeddings (fragrance_raw_full/embeddings.npy)')
_has_enr=os.path.exists(ENR_EMB_PATH); _has_mm=os.path.exists(MM_EMB_PATH)
if not _has_enr: print('INFO: enriched embeddings not found -- using raw')
if not _has_mm:  print('INFO: multimodal embeddings not found -- signal zeroed')
LOAD_MAP={'text_full':TEXT_EMB_PATH}
if _has_enr: LOAD_MAP['enriched_2k']=ENR_EMB_PATH
if _has_mm:  LOAD_MAP['mm_2k']=MM_EMB_PATH
_store={}
_pbar=tqdm(total=len(LOAD_MAP),desc='Loading embeddings',ncols=90,unit='file')
def _load(kv): k,p=kv; return k,np.load(p).astype(np.float32)
with ThreadPoolExecutor(max_workers=4) as pool:
    futs={pool.submit(_load,kv):kv[0] for kv in LOAD_MAP.items()}
    for fut in as_completed(futs):
        k,arr=fut.result(); _store[k]=arr
        _pbar.set_postfix_str(f'{k} {arr.shape}'); _pbar.update(1)
_pbar.close()
TEXT_EMBEDDINGS=_store['text_full']; MM_EMBEDDINGS=_store.get('mm_2k'); ENR_EMBEDDINGS=_store.get('enriched_2k')
_csv_cands=[os.path.join(W2_ARTIFACTS,'vibescent_enriched_2k_v2.csv'),
            os.path.join(W2_ARTIFACTS,'vibescent_enriched_500_v2.csv'),
            os.path.join(REPO_DIR,'data','vibescent_unified.csv')]
FRAGRANCE_DF=None
for _c in _csv_cands:
    if os.path.exists(_c): FRAGRANCE_DF=pd.read_csv(_c); print(f'DF: {_c}  shape={FRAGRANCE_DF.shape}'); break
if FRAGRANCE_DF is None: raise FileNotFoundError('No fragrance DataFrame. Run Week 2 first.')
if 'fragrance_id' not in FRAGRANCE_DF.columns: FRAGRANCE_DF.insert(0,'fragrance_id',FRAGRANCE_DF.index.astype(str))
if ENR_EMBEDDINGS is not None and len(ENR_EMBEDDINGS)==len(FRAGRANCE_DF):
    ACTIVE_TEXT_EMBEDDINGS=ENR_EMBEDDINGS; print('Using enriched embeddings')
elif len(TEXT_EMBEDDINGS)==len(FRAGRANCE_DF):
    ACTIVE_TEXT_EMBEDDINGS=TEXT_EMBEDDINGS; print('Using raw embeddings')
else:
    _n=min(len(TEXT_EMBEDDINGS),len(FRAGRANCE_DF))
    ACTIVE_TEXT_EMBEDDINGS=TEXT_EMBEDDINGS[:_n]; FRAGRANCE_DF=FRAGRANCE_DF.iloc[:_n].reset_index(drop=True)
    if MM_EMBEDDINGS is not None: MM_EMBEDDINGS=MM_EMBEDDINGS[:_n]
    print(f'WARNING: aligned to {_n} rows')
print(f'Corpus: {len(ACTIVE_TEXT_EMBEDDINGS)} x {ACTIVE_TEXT_EMBEDDINGS.shape[1]}  ({time.perf_counter()-_t0:.1f}s)')

In [ ]:
# Stage 3: Load all models -- text embedder, MM embedder, Neil CNN, optional reranker
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import os,subprocess,time; import numpy as np; import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
_t0=time.perf_counter()
TEXT_EMB_MODEL_NAME='Qwen/Qwen3-Embedding-8B'
MM_EMB_MODEL_NAME='Qwen/Qwen3-VL-Embedding-8B'
print(f'Loading {TEXT_EMB_MODEL_NAME} ...')
_bar=tqdm(['Cache','Load','GPU','Warmup'],desc='Text embedder',ncols=90,unit='step')
try:
    text_embedder=SentenceTransformer(TEXT_EMB_MODEL_NAME,trust_remote_code=True,
        model_kwargs={'torch_dtype':torch.float16,'attn_implementation':'flash_attention_2'})
except Exception:
    text_embedder=SentenceTransformer(TEXT_EMB_MODEL_NAME,trust_remote_code=True,
        model_kwargs={'torch_dtype':torch.float16})
_bar.update(3)
with torch.cuda.amp.autocast(): _test=text_embedder.encode(['test'],normalize_embeddings=True)
TEXT_EMB_DIM=_test.shape[1]; _bar.update(1); _bar.close()
print(f'Text embedder ready. dim={TEXT_EMB_DIM}')
if ACTIVE_TEXT_EMBEDDINGS.shape[1]!=TEXT_EMB_DIM:
    _d=min(ACTIVE_TEXT_EMBEDDINGS.shape[1],TEXT_EMB_DIM)
    ACTIVE_TEXT_EMBEDDINGS=ACTIVE_TEXT_EMBEDDINGS[:,:_d]
    _n=np.linalg.norm(ACTIVE_TEXT_EMBEDDINGS,axis=1,keepdims=True)
    ACTIVE_TEXT_EMBEDDINGS=(ACTIVE_TEXT_EMBEDDINGS/np.where(_n==0,1,_n)).astype(np.float32)
def _embed_text(t):
    with torch.cuda.amp.autocast():
        v=text_embedder.encode([t],normalize_embeddings=True,batch_size=1,show_progress_bar=False)
    return v[:,:TEXT_EMB_DIM].astype(np.float32)
MM_EMBEDDER=None
if MM_EMBEDDINGS is not None and GPU_TIER in('A100','L4'):
    print(f'Loading {MM_EMB_MODEL_NAME} ...')
    try:
        from vibescents.qwen3_vl_embedding import Qwen3VLMultimodalEmbedder
        MM_EMBEDDER=Qwen3VLMultimodalEmbedder(model_name=MM_EMB_MODEL_NAME,model_kwargs={'torch_dtype':torch.float16})
        _mm_test=np.array(MM_EMBEDDER.embed_query('warmup'),dtype=np.float32)
        MM_EMB_DIM=len(_mm_test)
        if MM_EMBEDDINGS.shape[1]!=MM_EMB_DIM:
            _d=min(MM_EMBEDDINGS.shape[1],MM_EMB_DIM); MM_EMBEDDINGS=MM_EMBEDDINGS[:,:_d]
            _n=np.linalg.norm(MM_EMBEDDINGS,axis=1,keepdims=True)
            MM_EMBEDDINGS=(MM_EMBEDDINGS/np.where(_n==0,1,_n)).astype(np.float32)
        print(f'MM embedder ready. dim={MM_EMB_DIM}')
    except Exception as _e: print(f'WARNING: MM embedder: {_e}'); MM_EMBEDDER=None
else: print(f'Skipping MM (MM_EMBEDDINGS={MM_EMBEDDINGS is not None}, GPU={GPU_TIER})')
NEIL_CNN=None
print('Loading Neil CNN-CLIP Hybrid...')
_nb=tqdm(['Checkout','Load ckpt','Warmup'],desc='Neil CNN',ncols=90,unit='step')
for _mf in ['models/cnn_clip_hybrid.py','models/clip_standalone.py']:
    subprocess.run(['git','-C',REPO_DIR,'checkout','origin/Image_Processing','--',_mf],capture_output=True)
_nb.update(1)
_ckpt=None
for _cp in ['checkpoints/hybrid/best.pt','checkpoints/cnn/best.pt']:
    _a=os.path.join(REPO_DIR,_cp)
    if os.path.isfile(_a): _ckpt=_a; break
    _r=subprocess.run(['git','-C',REPO_DIR,'checkout','origin/Image_Processing','--',_cp],capture_output=True)
    if _r.returncode==0 and os.path.isfile(_a): _ckpt=_a; break
_nb.update(1)
if _ckpt:
    try:
        import sys as _sys; _sys.path.insert(0,os.path.join(REPO_DIR,'models'))
        try: from cnn_clip_hybrid import CNNCLIPHybrid
        except ImportError:
            import torch.nn as nn; from transformers import CLIPModel
            class CNNCLIPHybrid(nn.Module):
                def __init__(self):
                    super().__init__()
                    import torchvision.models as tvm
                    _res=tvm.resnet50(weights=None)
                    self.cnn_backbone=nn.Sequential(*list(_res.children())[:-1])
                    _clip=CLIPModel.from_pretrained('openai/clip-vit-large-patch14')
                    self.clip_vision=_clip.vision_model; self.clip_proj=_clip.visual_projection
                    for p in list(self.clip_vision.parameters())+list(self.clip_proj.parameters()): p.requires_grad=False
                    self.fusion=nn.Sequential(nn.Linear(2048+768,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,256))
                    self.formal_head=nn.Linear(256,3); self.season_head=nn.Linear(256,4)
                    self.gender_head=nn.Linear(256,3); self.time_head=nn.Linear(256,2); self.frequency_head=nn.Linear(256,2)
                def forward(self,x):
                    f=self.fusion(torch.cat([self.cnn_backbone(x).flatten(1),self.clip_proj(self.clip_vision(pixel_values=x).pooler_output)],1))
                    return{'formal':self.formal_head(f),'season':self.season_head(f),'gender':self.gender_head(f),'time':self.time_head(f),'frequency':self.frequency_head(f)}
        NEIL_CNN=CNNCLIPHybrid()
        _st=torch.load(_ckpt,map_location='cpu',weights_only=True)
        NEIL_CNN.load_state_dict(_st.get('model_state_dict',_st),strict=False)
        NEIL_CNN.to(DEVICE).eval()
        with torch.no_grad(): _o=NEIL_CNN(torch.zeros(1,3,224,224,device=DEVICE))
        assert 'formal' in _o and 'season' in _o
        _nb.update(1); _nb.close(); print(f'Neil CNN loaded: {_ckpt}')
    except Exception as _e: _nb.close(); print(f'WARNING: Neil CNN: {_e}'); NEIL_CNN=None
else: _nb.close(); print('WARNING: No Neil checkpoint found')
RERANKER=None
if SHIP_RERANKER:
    try:
        from vibescents.reranker import GeminiReranker
        RERANKER=GeminiReranker(); print('Reranker ready')
    except Exception as _e: print(f'WARNING: Reranker: {_e}')
print(f'Models loaded in {time.perf_counter()-_t0:.1f}s')
print(f'  text={TEXT_EMB_MODEL_NAME}  MM={"ok" if MM_EMBEDDER else "None"}')
print(f'  Neil={"ok" if NEIL_CNN else "None"}  reranker={"ok" if RERANKER else "None"}')

In [ ]:
# Stage 4: Build VibeScentEngine (4-signal fusion, implements RecommendationEngine protocol)
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import base64,io,time; import numpy as np; import pandas as pd; import torch
from vibescents.backend_app import RecommendationEngine
from vibescents.schemas import RecommendRequest,RecommendResponse,FragranceRecommendation,RetrievalCandidate
from vibescents.image_preprocess import decode_b64_to_cnn_tensor
from vibescents.image_scoring import score_candidate_pool
_TOPK=50; _RRK=10; _FINAL=5

def _occ_text(ctx):
    parts=[p for p in[ctx.eventType,ctx.timeOfDay,ctx.mood,ctx.customNotes]if p]
    return', '.join(parts) if parts else'general occasion'

def _mm_norm(a):
    lo,hi=a.min(),a.max()
    return np.ones_like(a)*0.5 if hi-lo<1e-9 else(a-lo)/(hi-lo)

def _struct_score(df):
    cols=[c for c in('formality','fresh_warm')if c in df.columns]
    if not cols: return np.ones(len(df),dtype=np.float32)*0.5
    v=np.zeros(len(df),dtype=np.float32)
    for c in cols: v+=df[c].fillna(0.5).values.astype(np.float32)
    return(v/len(cols)).clip(0,1)

class VibeScentEngine:
    def __init__(self,*,fragrance_df,text_embeddings,mm_embeddings,fusion_weights,neil_cnn,reranker=None,locked_cache=None):
        self._df=fragrance_df.reset_index(drop=True)
        self._text=text_embeddings; self._mm=mm_embeddings; self._w=fusion_weights
        self._cnn=neil_cnn; self._rr=reranker; self._cache=locked_cache or{}
        self._fid_idx={str(fid):i for i,fid in enumerate(self._df.get('fragrance_id',self._df.index))}

    def recommend(self,*,request,image_bytes):
        t0=time.perf_counter()
        ctx=request.context
        cache_key=f'{ctx.eventType}|{ctx.timeOfDay}|{ctx.mood}'
        if cache_key in self._cache:
            return RecommendResponse(recommendations=[FragranceRecommendation(**r)for r in self._cache[cache_key]['recommendations']])
        occ=_occ_text(ctx)
        q_text=_embed_text(occ)
        sig_text=(self._text@q_text.T).squeeze(1)
        if self._mm is not None and MM_EMBEDDER is not None:
            try:
                from PIL import Image
                _pil=Image.open(io.BytesIO(image_bytes)).convert('RGB')
                _buf=io.BytesIO(); _pil.save(_buf,format='JPEG'); _buf.seek(0)
                _b64=base64.b64encode(_buf.read()).decode()
                q_mm=np.array(MM_EMBEDDER.embed_query(occ,image_b64=_b64),dtype=np.float32).reshape(1,-1)
                q_mm=q_mm[:,:self._mm.shape[1]]
                q_mm/=(np.linalg.norm(q_mm,axis=1,keepdims=True)+1e-9)
                sig_mm_raw=(self._mm@q_mm.T).squeeze(1)
                sig_mm=np.zeros(len(self._df),dtype=np.float32)
                sig_mm[:len(sig_mm_raw)]=sig_mm_raw
            except Exception: sig_mm=np.zeros(len(self._df),dtype=np.float32)
        else: sig_mm=np.zeros(len(self._df),dtype=np.float32)
        if self._cnn is not None:
            try:
                _t=decode_b64_to_cnn_tensor(request.image,request.mimeType)
                sig_img=score_candidate_pool(self._df,self._cnn,_t,device=DEVICE)
            except Exception: sig_img=np.zeros(len(self._df),dtype=np.float32)
        else: sig_img=np.zeros(len(self._df),dtype=np.float32)
        sig_s=_struct_score(self._df)
        wt,wm,wi,ws=self._w.get('text',.3),self._w.get('multimodal',.25),self._w.get('image',.3),self._w.get('structured',.15)
        fused=wt*_mm_norm(sig_text)+wm*_mm_norm(sig_mm)+wi*_mm_norm(sig_img)+ws*_mm_norm(sig_s)
        top_idx=np.argpartition(-fused,min(_TOPK,len(fused)-1))[:_TOPK]
        top_idx=top_idx[np.argsort(-fused[top_idx])]
        if self._rr is not None:
            rr_idx=top_idx[:_RRK]
            cands=[RetrievalCandidate(fragrance_id=str(self._df.iloc[i].get('fragrance_id',i)),
                name=str(self._df.iloc[i].get('name','')),brand=str(self._df.iloc[i].get('brand','')),
                retrieval_text=str(self._df.iloc[i].get('retrieval_text','')),baseline_score=float(fused[i]))
                for i in rr_idx]
            try:
                rr=self._rr.rerank(query=occ,candidates=cands,image_bytes=image_bytes)
                fin_idx=[self._fid_idx.get(r.fragrance_id,rr_idx[j])for j,r in enumerate(rr.results[:_FINAL])]
                fin_scores=[r.overall_score for r in rr.results[:_FINAL]]
                fin_reasons=[r.explanation for r in rr.results[:_FINAL]]
            except Exception as _e:
                print(f'  Reranker failed ({_e}) -- using fused')
                fin_idx=list(top_idx[:_FINAL]); fin_scores=[float(fused[i])for i in fin_idx]; fin_reasons=[None]*len(fin_idx)
        else:
            fin_idx=list(top_idx[:_FINAL]); fin_scores=[float(fused[i])for i in fin_idx]; fin_reasons=[None]*len(fin_idx)
        recs=[]
        for rank,(idx,score,reason) in enumerate(zip(fin_idx,fin_scores,fin_reasons),1):
            row=self._df.iloc[idx]
            name=str(row.get('name',f'Fragrance #{idx}')); house=str(row.get('brand','Unknown'))
            notes=[]
            for col in('top_notes','middle_notes','base_notes','main_accords'):
                if col in row and pd.notna(row[col]): notes+=[n.strip()for n in str(row[col]).split(',')if n.strip()]
            notes=list(dict.fromkeys(notes))[:8]
            if reason is None:
                vibe=str(row.get('vibe_sentence',''))if pd.notna(row.get('vibe_sentence',''))else''
                reason=f'{vibe}  (score:{score:.3f})'if vibe else f'Top {rank} by 4-signal fusion (score:{score:.3f})'
            occ_lbl=str(row.get('likely_occasion',occ))
            if not occ_lbl or occ_lbl=='nan': occ_lbl=occ
            recs.append(FragranceRecommendation(rank=rank,name=name,house=house,
                score=round(float(score),4),notes=notes,reasoning=reason,occasion=occ_lbl))
        print(f'  recommend {time.perf_counter()-t0:.2f}s  top:{recs[0].name if recs else"none"}')
        return RecommendResponse(recommendations=recs)

ENGINE=VibeScentEngine(fragrance_df=FRAGRANCE_DF,text_embeddings=ACTIVE_TEXT_EMBEDDINGS,
    mm_embeddings=MM_EMBEDDINGS,fusion_weights=FUSION_WEIGHTS,neil_cnn=NEIL_CNN,reranker=RERANKER)
print(f'VibeScentEngine built. corpus={len(FRAGRANCE_DF)} fragrances')
print(f'  weights={FUSION_WEIGHTS}')

In [ ]:
# Stage 5: Start FastAPI server (uvicorn in background thread)
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import threading,time,httpx,nest_asyncio,asyncio
nest_asyncio.apply()
import uvicorn
from vibescents.backend_app import create_app
APP=create_app(engine=ENGINE)
_cfg=uvicorn.Config(app=APP,host=FASTAPI_HOST,port=FASTAPI_PORT,log_level='warning',loop='asyncio',access_log=False)
_srv=uvicorn.Server(config=_cfg)
def _run(): asyncio.run(_srv.serve())
threading.Thread(target=_run,daemon=True,name='uvicorn').start()
_dl=time.time()+30
while time.time()<_dl:
    try:
        if httpx.get(f'{_BASE_URL}/healthz',timeout=2.0).status_code==200: break
    except Exception: pass
    time.sleep(0.3)
else: raise RuntimeError('FastAPI did not start in 30s')
print(f'FastAPI ready at {_BASE_URL}')
print(f'Health: {httpx.get(f"{_BASE_URL}/healthz").json()}')

In [ ]:
# Stage 6: Dual tunnel -- Cloudflare primary + ngrok backup
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import re,subprocess,time
CLOUDFLARE_URL=None; NGROK_URL=None
print('Starting Cloudflare Tunnel ...')
_cf=subprocess.Popen([CLOUDFLARED_BIN,'tunnel','--url',f'http://localhost:{FASTAPI_PORT}','--no-autoupdate'],
    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
_pat=re.compile(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com')
_dl=time.time()+60
while time.time()<_dl:
    line=_cf.stdout.readline()
    if not line: time.sleep(0.2); continue
    m=_pat.search(line)
    if m: CLOUDFLARE_URL=m.group(0); break
if CLOUDFLARE_URL: print(f'Cloudflare: {CLOUDFLARE_URL}')
else: print('WARNING: Cloudflare URL not found')
print('Starting ngrok backup ...')
try:
    from pyngrok import ngrok as _ng
    _tun=_ng.connect(FASTAPI_PORT,bind_tls=True)
    NGROK_URL=_tun.public_url
    if NGROK_URL.startswith('http://'): NGROK_URL='https://'+NGROK_URL[7:]
    print(f'ngrok:      {NGROK_URL}')
except Exception as _e: print(f'ngrok unavailable: {_e}')
print(); print('='*70); print('TUNNEL SUMMARY')
ACTIVE_BACKEND_URL=CLOUDFLARE_URL or NGROK_URL
if CLOUDFLARE_URL: print(f'  PRIMARY: {CLOUDFLARE_URL}'); print(f'  -> NEXT_PUBLIC_BACKEND_URL={CLOUDFLARE_URL}')
else: print('  PRIMARY (Cloudflare): NOT AVAILABLE')
if NGROK_URL: print(f'  BACKUP:  {NGROK_URL}')
else: print('  BACKUP  (ngrok): NOT AVAILABLE')
print('='*70)
if ACTIVE_BACKEND_URL is None: print('WARNING: No public tunnel -- local only')
else: print(f'ACTIVE_BACKEND_URL={ACTIVE_BACKEND_URL}')

In [ ]:
# Stage 7: Locked demo cache -- pre-generate 5 deterministic responses
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import json,os,time,httpx
from tqdm.auto import tqdm
_WHITE_PNG_B64=('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk'
                '+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg==')
LOCKED_CASES=[
    {'key':'wedding|evening','request':{'image':_WHITE_PNG_B64,'mimeType':'image/png','context':{'eventType':'wedding','timeOfDay':'evening','mood':'romantic','customNotes':None}}},
    {'key':'business meeting|morning','request':{'image':_WHITE_PNG_B64,'mimeType':'image/png','context':{'eventType':'business meeting','timeOfDay':'morning','mood':'confident','customNotes':None}}},
    {'key':'beach vacation|afternoon','request':{'image':_WHITE_PNG_B64,'mimeType':'image/png','context':{'eventType':'beach vacation','timeOfDay':'afternoon','mood':'relaxed','customNotes':None}}},
    {'key':'date night|evening','request':{'image':_WHITE_PNG_B64,'mimeType':'image/png','context':{'eventType':'date night','timeOfDay':'evening','mood':'sensual','customNotes':None}}},
    {'key':'casual day out|afternoon','request':{'image':_WHITE_PNG_B64,'mimeType':'image/png','context':{'eventType':'casual day out','timeOfDay':'afternoon','mood':'happy','customNotes':None}}},
]
local_cache_path=os.path.join(REPO_DIR,'artifacts','week4','locked_responses.json')
os.makedirs(os.path.dirname(local_cache_path),exist_ok=True)
locked_responses={}
_pb=tqdm(LOCKED_CASES,desc='Generating locked responses',ncols=90,unit='case')
for case in _pb:
    key=case['key']; _pb.set_postfix_str(key)
    try:
        resp=httpx.post(f'{_BASE_URL}/recommend',json=case['request'],timeout=120.0)
        resp.raise_for_status()
        locked_responses[key]=resp.json()  # NOTE: .json() -- parentheses required
        print(f"  '{key}' -> {len(locked_responses[key].get('recommendations',[]))} recs")
    except Exception as _e: print(f"  WARNING: '{key}' failed: {_e}")
_pb.close()
ENGINE._cache.update(locked_responses)
print(f'Injected {len(locked_responses)} responses into engine cache')
_payload=json.dumps(locked_responses,indent=2)
open(local_cache_path,'w').write(_payload)
try: open(os.path.join(W4_ARTIFACTS,'locked_responses.json'),'w').write(_payload); print(f'Saved to local+Drive')
except Exception: print(f'Saved to local only')

In [ ]:
# Stage 8: Generate Frontend Patch (frontend_patch.diff for Darren)
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import os
_patch_local=os.path.join(REPO_DIR,'artifacts','week4','frontend_patch.diff')
_patch_drive=os.path.join(W4_ARTIFACTS,'frontend_patch.diff')
os.makedirs(os.path.dirname(_patch_local),exist_ok=True)
FRONTEND_PATCH = '--- a/app/api/recommend/route.ts\n+++ b/app/api/recommend/route.ts\n@@ -1,19 +1,48 @@\n import { NextRequest, NextResponse } from "next/server";\n import type { RecommendRequest, RecommendResponse } from "@/lib/types";\n\n-export async function POST(req: NextRequest) {\n-  const body: RecommendRequest = await req.json();\n-  await new Promise((resolve) => setTimeout(resolve, 1500));\n-  return NextResponse.json({ recommendations: [] });\n+const BACKEND_URL = process.env.NEXT_PUBLIC_BACKEND_URL;\n+const LOCKED_CACHE: Record<string, RecommendResponse> = {};\n+function cacheKey(b: RecommendRequest): string {\n+  return `${b.context.eventType ?? ""}|${b.context.timeOfDay ?? ""}|${b.context.mood ?? ""}`;\n }\n+export async function POST(req: NextRequest) {\n+  const body: RecommendRequest = await req.json();\n+  const key = cacheKey(body);\n+  if (BACKEND_URL) {\n+    try {\n+      const r = await fetch(`${BACKEND_URL}/recommend`, {\n+        method: "POST", headers: { "Content-Type": "application/json" },\n+        body: JSON.stringify(body), signal: AbortSignal.timeout(90_000),\n+      });\n+      if (r.ok) { const data = await r.json(); LOCKED_CACHE[key]=data; return NextResponse.json(data); }\n+    } catch(e) { console.warn("[VibeScent] Backend unreachable:", e); }\n+  }\n+  if (LOCKED_CACHE[key]) return NextResponse.json({ ...LOCKED_CACHE[key], _source: "cache" });\n+  return NextResponse.json({ error: "Backend unreachable and no cached response." }, { status: 503 });\n+}\n'
open(_patch_local,'w').write(FRONTEND_PATCH)
try: open(_patch_drive,'w').write(FRONTEND_PATCH); print(f'Patch saved: local+Drive')
except Exception: print(f'Patch saved: local only')
print(f'  Apply with: git apply {_patch_local}')
print(f'  Darren .env.local: NEXT_PUBLIC_BACKEND_URL={ACTIVE_BACKEND_URL or "<paste-url>"}')

In [ ]:
# Stage 9: Smoke Test -- POST to local, Cloudflare, ngrok; validate contract
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import httpx
_WHITE_PNG_B64=('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk'
                '+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg==')
_PAYLOAD={'image':_WHITE_PNG_B64,'mimeType':'image/png','context':{'eventType':'casual day out','timeOfDay':'afternoon','mood':'happy','customNotes':None}}
_REQUIRED={'rank','name','house','score','notes','reasoning','occasion'}
def _validate(data,url):
    recs=data.get('recommendations',[])
    assert isinstance(recs,list) and len(recs)>0,f'empty recommendations from {url}'
    for i,r in enumerate(recs):
        miss=_REQUIRED-set(r.keys())
        assert not miss,f'rec[{i}] missing: {miss}'
        assert isinstance(r['rank'],int) and r['rank']>=1
    print(f'  PASS ({url}) -- {len(recs)} recs')
    for r in recs[:3]: print(f'    #{r["rank"]} {r["name"]} | {r["house"]} | {r["score"]:.4f}')
    return True
_ok=True
print('Test 1: Local server')
try:
    _r=httpx.post(f'{_BASE_URL}/recommend',json=_PAYLOAD,timeout=120.0)
    _r.raise_for_status(); _validate(_r.json(),_BASE_URL)
except Exception as _e: print(f'  FAIL: {_e}'); _ok=False
if CLOUDFLARE_URL:
    print(f'Test 2: Cloudflare ({CLOUDFLARE_URL})')
    try:
        _r2=httpx.post(f'{CLOUDFLARE_URL}/recommend',json=_PAYLOAD,timeout=60.0,follow_redirects=True)
        _r2.raise_for_status(); _validate(_r2.json(),CLOUDFLARE_URL)
    except Exception as _e2: print(f'  WARNING: {_e2}')
if NGROK_URL:
    print(f'Test 3: ngrok ({NGROK_URL})')
    try:
        _r3=httpx.post(f'{NGROK_URL}/recommend',json=_PAYLOAD,timeout=60.0,follow_redirects=True)
        _r3.raise_for_status(); _validate(_r3.json(),NGROK_URL)
    except Exception as _e3: print(f'  WARNING: {_e3}')
print(f'Health: {httpx.get(f"{_BASE_URL}/healthz").json()}')
print('All smoke tests PASSED' if _ok else 'WARNING: some tests failed')

In [ ]:
# Stage 10: Pre-Demo Runbook
PIPELINE_VERSION = globals().get('PIPELINE_VERSION','w4v1')
import os
_D='='*70
print(_D); print('  VIBESCENT WEEK 4 DEMO -- DAY-OF RUNBOOK'); print(_D)
print('''
BEFORE PRESENTING
-----------------
1. Colab Pro A100 GPU selected (Runtime > Change runtime type)
2. Verify Drive: !ls /content/drive/MyDrive/vibescent/week3/
   Expected: best_weights.json  reranker_decision.json
3. Verify W2 artifacts: !ls /content/drive/MyDrive/vibescent/fragrance_raw_full/
   Expected: embeddings.npy  manifest.json
4. Colab Secrets (key icon): GEMINI_API_KEY, VOYAGE_API_KEY
5. Runtime > Run all  (~10-15 min first run)
6. Both tunnel URLs printed in Stage 6 -- copy PRIMARY to Darren
''')
print('SHARING TUNNEL URL WITH DARREN')
print('-'*40)
print('Tell Darren to add to frontend .env.local:')
print(f'  NEXT_PUBLIC_BACKEND_URL={CLOUDFLARE_URL or "<paste Cloudflare URL>"}')
print()
print('IF TUNNEL DROPS')
print('-'*40)
print('A: Switch to ngrok backup URL (Stage 6)')
print('B: Re-run Stage 6 cell only -- new tunnel instantly')
print('C: Offline cached demo -- 5 rehearsed cases work with NO tunnel:')
for _c in LOCKED_CASES: print(f'   * {_c["key"]}')
print()
print('PIPELINE SUMMARY')
print('-'*40)
print(f'  version      : {PIPELINE_VERSION}')
print(f'  GPU tier     : {GPU_TIER}')
print(f'  corpus       : {len(FRAGRANCE_DF)} fragrances')
print(f'  weights      : {FUSION_WEIGHTS}')
print(f'  reranker     : {"ON (Gemini)" if SHIP_RERANKER else "OFF -- fused top-5"}')
print(f'  text model   : {TEXT_EMB_MODEL_NAME}')
print(f'  mm model     : {MM_EMB_MODEL_NAME}')
print(f'  PRIMARY URL  : {CLOUDFLARE_URL or "NOT SET"}')
print(f'  BACKUP URL   : {NGROK_URL or "NOT SET"}')
print()
print('ARTIFACT LOCATIONS')
print('-'*40)
print(f'  Locked cache : {local_cache_path}')
print(f'  Drive cache  : {os.path.join(W4_ARTIFACTS,"locked_responses.json")}')
print(f'  Frontend diff: {os.path.join(REPO_DIR,"artifacts","week4","frontend_patch.diff")}')
print(f'  Drive week4  : {W4_ARTIFACTS}')
print()
print(f'Health check: httpx.get("{_BASE_URL}/healthz").json()')
print(_D); print('  BACKEND IS READY. GOOD LUCK!'); print(_D)